In [2]:
import requests
import sqlite3

url = "http://universities.hipolabs.com/search?country=Brazil"

response = requests.get(url)
response.raise_for_status()

universities = response.json()
print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


In [3]:
'''
(1) - Modelagem proposta:

Tabela Universities:
- id (int, primary key)
- name (string)
- country (string)
- state-province (string)
- alpha_two_code (string)

Tabela Sites:
- id (int, primary key)
- university_id (int, foreign key)
- url (string)

Justificativa: Separar em duas tabelas evita repetir os dados da universidade quando ela tem vários sites.
'''

'\n(1) - Modelagem proposta:\n\nTabela Universities:\n- id (int, primary key)\n- name (string)\n- country (string)\n- state-province (string)\n- alpha_two_code (string)\n\nTabela Sites:\n- id (int, primary key)\n- university_id (int, foreign key)\n- url (string)\n\nJustificativa: Separar em duas tabelas evita repetir os dados da universidade quando ela tem vários sites.\n'

In [4]:
# CRIAR BANCO DE DADOS E TABELAS
conn = sqlite3.connect('universidades.db')
cursor = conn.cursor()

# Cria tabela de universidades
cursor.execute('''
CREATE TABLE IF NOT EXISTS universities (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    country TEXT NOT NULL,
    state_province TEXT,
    alpha_two_code TEXT
)
''')

# Cria tabela de websites
cursor.execute('''
CREATE TABLE IF NOT EXISTS websites (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    university_id INTEGER NOT NULL,
    url TEXT NOT NULL,
    FOREIGN KEY (university_id) REFERENCES universities (id)
)
''')

conn.commit()
print("Tabelas criadas com sucesso!")

Tabelas criadas com sucesso!


In [5]:
'''
(2) - Melhor estratégia para vários países:

Usar um loop processando um país por vez:
'''

countries = ["Brazil", "Argentina", "USA", "Australia"]

for country in countries:
    print(f"Processando {country}...")

    url = f"http://universities.hipolabs.com/search?country={country}"
    response = requests.get(url)

    universities = response.json()

    '''
    Em seguida, adicionamos os dados obtidos ao banco de dados
    '''

    for uni in universities:
        name = uni.get('name')
        if not name:
            continue

        state_province = uni.get('state-province')
        alpha_two_code = uni.get('alpha_two_code')
        websites_list = uni.get('web_pages', [])

        # Insere a universidade
        cursor.execute('''
        INSERT INTO universities (name, country, state_province, alpha_two_code)
        VALUES (?, ?, ?, ?)
        ''', (name, country, state_province, alpha_two_code))

        # Pega o ID gerado
        uni_id = cursor.lastrowid

        # Insere os websites
        for site in websites_list:
            if site:  # Só insere se tiver URL
                cursor.execute('''
                INSERT INTO websites (university_id, url)
                VALUES (?, ?)
                ''', (uni_id, site))

    conn.commit()
    print(f"Universidades de {country} salvas no banco!")

Processando Brazil...
Universidades de Brazil salvas no banco!
Processando Argentina...
Universidades de Argentina salvas no banco!
Processando USA...
Universidades de USA salvas no banco!
Processando Australia...
Universidades de Australia salvas no banco!
